In [1]:
import numpy as np
import pandas as pd
import requests
import duckdb
import os
import s3fs
import sys
sys.path.append('../outils')

import fonctions_nomenclature as fnom
import fonctions_preparation_table as fprep

np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"

In [2]:
FILE_KEY_S3 = "t1_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t1_muni = pd.read_csv(file_in, sep=",")

t1_muni.head()

/tmp/ipykernel_5797/3544469253.py:5: DtypeWarning: Columns (1: Code département, 3: Code commune, 127: Nuance liste 9, 128: Libellé abrégé de liste 9, 129: Libellé de liste 9, 131: % Voix/inscrits 9, 132: % Voix/exprimés 9, 140: Nuance liste 10, 141: Libellé abrégé de liste 10, 142: Libellé de liste 10, 144: % Voix/inscrits 10, 145: % Voix/exprimés 10, 153: Nuance liste 11, 154: Libellé abrégé de liste 11, 155: Libellé de liste 11, 157: % Voix/inscrits 11, 158: % Voix/exprimés 11, 166: Nuance liste 12, 167: Libellé abrégé de liste 12, 168: Libellé de liste 12, 170: % Voix/inscrits 12, 171: % Voix/exprimés 12, 179: Nuance liste 13, 180: Libellé abrégé de liste 13, 181: Libellé de liste 13, 183: % Voix/inscrits 13, 184: % Voix/exprimés 13) have mixed types. Specify dtype option on import or set low_memory=False.
  t1_muni = pd.read_csv(file_in, sep=",")


,Unnamed: 0,Code département,Libellé département,Code commune,Libellé commune,Inscrits,Votants,% Votants,Abstentions,% Abstentions,...,Sexe candidat 13,Nuance liste 13,Libellé abrégé de liste 13,Libellé de liste 13,Voix 13,% Voix/inscrits 13,% Voix/exprimés 13,Elu 13,Sièges au CM 13,Sièges au CC 13
0,0,1,Ain,1001,L'Abergement-Clémenciat,679,374,"55,08%",305,"44,92%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,Ain,1002,L'Abergement-de-Varey,246,206,"83,74%",40,"16,26%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1,Ain,1004,Ambérieu-en-Bugey,8806,4286,"48,67%",4520,"51,33%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1,Ain,1005,Ambérieux-en-Dombes,1362,592,"43,47%",770,"56,53%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1,Ain,1006,Ambléon,107,93,"86,92%",14,"13,08%",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
FILE_KEY_S3 = "t2_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    t2_muni = pd.read_csv(file_in, sep=",")

t2_muni.head()

In [ ]:
FILE_KEY_S3 = "bistrots_sirene.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    bistrots_sirene = pd.read_csv(file_in, sep=",")

bistrots_sirene.head()

In [ ]:
FILE_KEY_S3 = "filosofi.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    filosofi = pd.read_csv(file_in, sep=",")

filosofi.head()

## Explo et prep données bistrots 

In [ ]:
bistrots_sirene.describe()

In [ ]:
# Data prep Sirene ----------------------------
## Modifier le format de la dateDernierTraitementEtablissement (enlever l'heure) et de la dateCreationEtablissement 
bistrots_sirene["date_dernier_traitement"] = pd.to_datetime(bistrots_sirene.dateDernierTraitementEtablissement).dt.normalize()
bistrots_sirene["date_creation"] = pd.to_datetime(bistrots_sirene.dateCreationEtablissement).dt.normalize()
bistrots_sirene.head()

In [ ]:
bistrots_sirene["date_dernier_traitement"].describe()

In [ ]:
bistrots_sirene["date_creation"].describe()

In [ ]:
# Proposition code alternatif pour décomptes sans copie
cond_date = "2026-01-01"
print(bistrots_sirene["siret"].nunique())
print(bistrots_sirene[(bistrots_sirene['date_creation'] <= cond_date)]["siret"].nunique())
print(bistrots_sirene[(bistrots_sirene['date_creation'].isnull())]["siret"].nunique())

In [ ]:
bistrots_par_ville = (
    bistrots_sirene
    .assign(
        age=(
            pd.Timestamp("2026-03-01")
            - pd.to_datetime(
                bistrots_sirene["dateCreationEtablissement"],
                errors="coerce"
            )
        ).dt.days / 365.25
    )
    .groupby(["codeCommuneEtablissement"])
    .agg(
        nb_etablissements=("siret", "nunique"),
        age_moyen_bistrots=("age", "mean")
    )
    .reset_index()
    .sort_values("nb_etablissements", ascending=False)
)

In [ ]:
bistrots_par_ville.head(20)

## Explo et prep données municipales

On cherche à construire 3 indicateurs : 

1°) Le vote simili Attal: 2 si la nuance majoritaire au conseil municipal est sans étiquette, 1 si elle est de droite, 0 sinon 

2°) La majorité simili Attal : 2 si la couleur majoritaire au conseil municipal est sans étiquette, 1 si elle est de droite, 0 sinon (indépendamment de la liste arrivée en tête)

3°) Le vote d'extrême droite : 2 si la nuance majoritaire au conseil municipal est sans étiquette, 1 si la nuance majoritaire au conseil municipal est d'extrême droite, 0 sinon

Nous allons d'abord préparer la table du deuxième tour, qui contient uniquement les villes pour lesquelles il y a eu deux tours d'élection.
Ainsi, nous pourrons récupérer les codes communes de ces villes, pour les enlever de la table du premier tour et ne traiter que les villes n'ayant eu qu'un tour. 

PS : dans un deuxième temps, peut être qu'on reviendra sur ce traitement pour ajouter des indicateurs sur le nombre de tours, le nombre de listes etc. 

### Préparation de la table du deuxième tour
On commence par traiter les villes ayant eu deux tours aux élections municipales

In [ ]:
t2_muni.head(20)

In [ ]:
t2_muni.describe()

In [ ]:
t2_muni.columns #=> trop de colonnes, il faut sélectionner les colonnes pertinentes 

In [ ]:
# Synthèse de la table du deuxième tour ------------------------
t2_muni_long = fprep.synthese_table_muni(t2_muni)

In [ ]:
# Création d'une nomenclature associant une couleur à une nuance ------------

## Récupérer les nuances
label_nuance = pd.DataFrame({"Nuance": t2_muni_long["Nuance"].unique()})
# print(label_nuance)

nomenclature_couleur_nuance = fnom.classer_nuance(label_nuance) 
#print(nomenclature_couleur_nuance)

In [ ]:
# Création variable nombre de candidats
t2_muni_long['Nombre candidats'] = (
    t2_muni_long['Sièges au CM'].notna()
    .groupby(t2_muni_long['Code commune'])
    .transform('sum')
)

In [ ]:
# Création indicateur 1 ----------------------------------
t2_muni_ind1 = fprep.crea_ind1(t2_muni_long, nomenclature_couleur_nuance)
print(t2_muni_ind1['ind1'].value_counts()) 

In [ ]:
# Création indicateur 2 ----------------------------------
t2_muni_ind2 = fprep.crea_ind2(t2_muni_long, nomenclature_couleur_nuance)
print(t2_muni_ind2['ind2'].value_counts())

In [ ]:
# Création indicateur 3 ----------------------------------
t2_muni_ind3 = fprep.crea_ind3(t2_muni_ind1.drop(columns=["ind1"])) 
t2_muni_ind3['ind3'].value_counts()

In [ ]:
# Table synthétique contenant les 3 indicateurs --------
t2_muni_ind = (
    t2_muni_ind1
    .drop(columns=["Nuance_maj", "Couleur"])
    .merge(t2_muni_ind2[["Code commune", "ind2"]], on="Code commune", how="left")
    .merge(t2_muni_ind3[["Code commune", "ind3"]], on="Code commune", how="left")
)

t2_muni_ind["tour"] = 2

t2_muni_ind.head()

### Préparation de la table du premier tour

Informations sur T1 : 
- La table du premier tour contient les communes ayant eu deux tours aux élections, donc des communes déjà traitées dans T2. Il s'agit donc de sélectionner négativement les communes non encore traitées. 
- La table T1 a la même structure que la table T2 : les fonctions utilisées pour T2 peuvent l'être pour T1

In [ ]:
# Selection des villes n'ayant eu qu'un seul tour d'élection (en pratique : on enlève celles qui en ont eu 2)

## Récupérer les codes communes des communes ayant eu deux tours d'élection 
cc_2t = t2_muni_ind["Code commune"].unique()

## Mise au format des codes commune dans la table t1 : rajouter un 0 devant les codes commune avec 4 chiffres
t1_muni["Code commune"]= t1_muni["Code commune"].astype(str).str.zfill(5)
## Sélection inversée 
t1_muni_keep = t1_muni[~t1_muni["Code commune"].isin(cc_2t)] 

In [ ]:
# Vérification : a-t-on toutes les villes ? 
print(t1_muni["Code commune"].unique().shape[0]) # 34836
print(t1_muni_keep["Code commune"].unique().shape[0] +len(cc_2t)) #34836

In [ ]:
# Préparation de T1 à l'image de T2 
t1_muni_long = fprep.synthese_table_muni(t1_muni_keep)

## Vérif : je veux voir si le "nb de codes communes de T1 avant sélection (ie, ttes les communes)" == nb codes communes t1_keep + nb code commune t2, pour voir si mon code "synthese_table_muni" ne supprime pas des communes
print(len(t1_muni["Code commune"].unique()))
print(len(t1_muni_long["Code commune"].unique()) + len(cc_2t))


In [ ]:
# Création variable nombre de candidats => prend toutes les colonnes vides, il faut select celles ou CM est non vide ? 
t1_muni_long['Nombre candidats'] = (
    t1_muni_long['Sièges au CM'].notna()
    .groupby(t1_muni_long['Code commune'])
    .transform('sum')
)
t1_muni_long.describe()

In [ ]:
# Création de ind1
t1_muni_ind1 = fprep.crea_ind1(t1_muni_long, nomenclature_couleur_nuance)
print(t1_muni_ind1['ind1'].value_counts()) # beaucoup beaucoup de communes avec des listes sans étiquette ... 

In [ ]:
#Création de ind2
t1_muni_ind2 = fprep.crea_ind2(t1_muni_long, nomenclature_couleur_nuance)
print(t1_muni_ind2['ind2'].value_counts())

In [ ]:
#Création de ind3
t1_muni_ind3 = fprep.crea_ind3(t1_muni_ind1.drop(columns=["ind1"]))
print(t1_muni_ind3['ind3'].value_counts())

In [ ]:
# Jointure des trois tables d'indicateurs + ajout indicatrice tour 1 ou 2

t1_muni_ind = (
    t1_muni_ind1
    .drop(columns=["Nuance_maj", "Couleur"])
    .merge(t1_muni_ind2[["Code commune", "ind2"]], on="Code commune", how="left")
    .merge(t1_muni_ind3[["Code commune", "ind3"]], on="Code commune", how="left")
)

t1_muni_ind["tour"] = 1

t1_muni_ind.head()

### Empiler les tables des deux tours

In [ ]:
table_complete_muni = pd.concat([t1_muni_ind, t2_muni_ind], ignore_index=True)
table_complete_muni.head

In [ ]:
pd.crosstab(table_complete_muni['ind1'], table_complete_muni['ind3'])

In [ ]:
pd.crosstab(table_complete_muni['ind1'], table_complete_muni['Nombre candidats'])

In [ ]:
# TO SUPPR

#todo - j'utilise pour l'instant la copie de t2 en attendant la factorisation et complétion avec t1
#Warning ça ne sera pas une jointure entre les deux tours mais un empilement je pense, vu qu'on veut une table avec le code géo en identifiant
#table_complete_muni = t2_muni_ind.copy()

## Jointure des tables sur code geo 

In [ ]:
table_analyse = filosofi.merge(bistrots_par_ville, left_on="CODGEO", right_on="codeCommuneEtablissement", how="left")
print(filosofi.shape[0])
print(bistrots_par_ville["codeCommuneEtablissement"].nunique())
print(bistrots_par_ville.shape[0])
print(table_analyse.shape[0])

In [ ]:
table_analyse_muni = table_analyse.merge(table_complete_muni, left_on="CODGEO", right_on="Code commune", how="left") # Changer le sens de la jointure une fois que l'on a toutes les communes
print(table_analyse.shape[0])
print(table_analyse["CODGEO"].nunique())
print(table_complete_muni["Code commune"].nunique())
print(table_complete_muni.shape[0])
print(table_analyse_muni.shape[0])

In [ ]:
table_analyse_muni.head()

In [ ]:
FILE_KEY_OUT_S3 = "table_analyse_muni.csv"
FILE_PATH_OUT_S3 = BUCKET + "/" + FILE_KEY_OUT_S3

with fs.open(FILE_PATH_OUT_S3, 'w') as file_out:
    table_analyse_muni.to_csv(file_out)

# TODO data prep

- Filtrer les bistrots actifs à une date t que l'on déterminera [en cours, besoin de se renseigner sur la façon dont est remplie la variable dateDernierTraitementEtablissement] OK
- Agréger les bistrots Sirene en un compte de bistrots par commune OK
- Traiter les données des municipales pour obtenir un indicateur quanti du vote "simili-Attal" (majo présidentielle, en gros) et du vote RN [EN COURS]
    - Table T1 : [En cours]
        - Garder le nb de communes qui ont eu 1 seul tour uniquement (fait)
        - Traiter ces communes en factorisant le code de T2 dans des fonctions 
    - Table T2 : en cours + à vérifier : 
        - Regarder ce que fais la ligne .dropna(subset=["Nuance"]) => supprime les codes commune sans nuance renseignée. Cb de communes cela concerne ? comment gérer le pb ? garder que la colonne qui finit en 1 et mettre "autre nuance" ? Que signifie une absence de nuance ? une liste indépendante ? 
        - Après création de T1 grace à des fonctions ad hoc adaptées du traitement de T2, utiliser ces fonctions sur T2
    - Fusionner les tables T1 et T2 propres
- Sélectionner les variables d'intérêt dans chaque table pour en faire quelque chose de lisible (bistrots OK, filosofi Ok mais deuxième passe possible, Muni en cours)
- Fusionner les tables (partir des tables T1 et T2 fusionnées comme référence) Fait partiellement, en attente de T1